In [14]:
!pip install -q snowflake-connector-python pandas requests prefect

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.


In [15]:
import json
import uuid
import pandas as pd
import snowflake.connector

from pathlib import Path
from getpass import getpass
from IPython.display import display
from prefect import task, flow

ACCOUNT = "EMIEDLR-JD97213"
USER = "Aadarshp037"
PASSWORD = getpass("Snowflake password: ")

conn = snowflake.connector.connect(
    account=ACCOUNT,
    user=USER,
    password=PASSWORD,
    role="ACCOUNTADMIN",
    login_timeout=30,
    network_timeout=60
)

cur = conn.cursor()

cur.execute("""
SELECT CURRENT_ACCOUNT(),
       CURRENT_DATABASE(),
       CURRENT_ROLE()
""")

print("Connected:", cur.fetchone())

Snowflake password: ··········


/usr/local/lib/python3.13/dist-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


Connected: ('ZO15526', None, 'ACCOUNTADMIN')


In [16]:
cur.execute("""
CREATE WAREHOUSE IF NOT EXISTS FRESHCART_WH
WITH WAREHOUSE_SIZE='XSMALL'
AUTO_SUSPEND=60
AUTO_RESUME=TRUE
INITIALLY_SUSPENDED=TRUE
""")

cur.execute("USE WAREHOUSE FRESHCART_WH")

cur.execute("CREATE DATABASE IF NOT EXISTS FRESHCART_DB")
cur.execute("USE DATABASE FRESHCART_DB")

for schema in ["BRONZE", "SILVER", "DIM", "REF", "GOLD"]:
    cur.execute(f"CREATE SCHEMA IF NOT EXISTS {schema}")

cur.execute("""
CREATE FILE FORMAT IF NOT EXISTS BRONZE.FRESHCART_JSON
TYPE=JSON
STRIP_OUTER_ARRAY=TRUE
""")

cur.execute("""
CREATE STAGE IF NOT EXISTS BRONZE.RAW_STAGE
FILE_FORMAT=BRONZE.FRESHCART_JSON
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS BRONZE.ORDERS_RAW (
    RAW_ID NUMBER AUTOINCREMENT,
    SOURCE VARCHAR(20),
    BATCH_ID VARCHAR(50),
    PAYLOAD VARIANT,
    INGESTED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
""")

print("FreshCart infrastructure ready!")

FreshCart infrastructure ready!


In [17]:
WEB_API_DATA = [
    {
        "order_id": "ORD-1001",
        "customer_id": "CUST-01",
        "customer_email": "priya.k@example.com",
        "order_date": "2026-08-01",
        "amount": 1100,
        "currency": "INR",
        "updated_at": "2026-08-01T10:00:00"
    },
    {
        "order_id": "ORD-1002",
        "customer_id": "CUST-02",
        "customer_email": "arjun.k@yahoo.in",
        "order_date": "2026-08-20",
        "amount": 785,
        "currency": "INR",
        "updated_at": "2026-08-20T10:00:00"
    },
    {
        "order_id": "ORD-1003",
        "customer_id": "CUST-03",
        "customer_email": "meera.s@example.com",
        "order_date": "2026-08-10",
        "currency": "INR",
        "updated_at": "2026-08-10T09:00:00"
    },
    {
        "order_id": "ORD-1004",
        "customer_id": None,
        "customer_email": None,
        "order_date": "2026-08-11",
        "amount": 300,
        "currency": "INR",
        "updated_at": "2026-08-11T09:00:00"
    },
    {
        "order_id": "ORD-1005",
        "customer_id": "CUST-01",
        "customer_email": "priya.k@example.com",
        "order_date": "2026-08-20",
        "amount": 3800,
        "currency": "INR",
        "updated_at": "2026-08-20T11:00:00"
    },
    {
        "order_id": "ORD-1006",
        "customer_id": "CUST-03",
        "customer_email": "meera.s@example.com",
        "order_date": "2026-08-21",
        "amount": -50,
        "currency": "EUR",
        "updated_at": "2026-08-21T09:00:00"
    }
]

print("Website API records:", len(WEB_API_DATA))

Website API records: 6


In [18]:
MOBILE_API_DATA = [
    {
        "orderId": "ORD-1001",
        "customerId": "CUST-01",
        "email": "priya.k@example.com",
        "orderDate": "2026-08-01",
        "total": 1100,
        "currencyCode": "INR",
        "updatedAt": "2026-08-01T10:05:00"
    },
    {
        "orderId": "ORD-1003",
        "customerId": "CUST-03",
        "email": "meera.s@example.com",
        "orderDate": "2026-08-10",
        "total": 2100,
        "currencyCode": "INR",
        "updatedAt": "2026-08-10T10:30:00"
    }
]

print("Mobile API records:", len(MOBILE_API_DATA))

Mobile API records: 2


In [19]:
LANDING_DIR = Path("/content/freshcart_landing")
LANDING_DIR.mkdir(parents=True, exist_ok=True)

def extract_api_data():
    return WEB_API_DATA, MOBILE_API_DATA


def write_landing(records, source, batch_id):

    data = [
        {**record, "_batch_id": batch_id}
        for record in records
    ]

    file_path = LANDING_DIR / f"{source}_{batch_id}.json"

    with open(file_path, "w") as f:
        json.dump(data, f, indent=4)

    print(source, "records written:", len(data))

    return file_path


web_rows, mobile_rows = extract_api_data()

BATCH_ID = uuid.uuid4().hex[:12]

web_file = write_landing(web_rows, "web", BATCH_ID)
mobile_file = write_landing(mobile_rows, "mobile", BATCH_ID)

print("Batch ID:", BATCH_ID)

web records written: 6
mobile records written: 2
Batch ID: ee07ce1af1a3


In [20]:
def put_file(file_path):

    local_path = Path(file_path).resolve().as_posix()

    cur.execute(f"""
        PUT 'file://{local_path}'
        @FRESHCART_DB.BRONZE.RAW_STAGE
        AUTO_COMPRESS=TRUE
        OVERWRITE=TRUE
    """)

    result = cur.fetchall()
    print(result)


put_file(web_file)
put_file(mobile_file)

[('web_ee07ce1af1a3.json', 'web_ee07ce1af1a3.json.gz', 1726, 320, 'NONE', 'GZIP', 'UPLOADED', '')]


[('mobile_ee07ce1af1a3.json', 'mobile_ee07ce1af1a3.json.gz', 574, 224, 'NONE', 'GZIP', 'UPLOADED', '')]


In [21]:
def load_file(file_path, source):

    filename = Path(file_path).name

    cur.execute(f"""
        COPY INTO FRESHCART_DB.BRONZE.ORDERS_RAW
        (SOURCE, BATCH_ID, PAYLOAD, INGESTED_AT)

        FROM (
            SELECT
                '{source}',
                $1:_batch_id::VARCHAR,
                OBJECT_DELETE($1, '_batch_id'),
                CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
            FROM @FRESHCART_DB.BRONZE.RAW_STAGE
        )

        FILE_FORMAT=(
            FORMAT_NAME='FRESHCART_DB.BRONZE.FRESHCART_JSON'
        )

        PATTERN='.*{filename}.*'
        FORCE=TRUE
        ON_ERROR='ABORT_STATEMENT'
    """)

    print(source, cur.fetchall())


load_file(web_file, "web")
load_file(mobile_file, "mobile")

cur.execute("""
SELECT SOURCE, COUNT(*)
FROM BRONZE.ORDERS_RAW
WHERE BATCH_ID=%s
GROUP BY SOURCE
""", (BATCH_ID,))

print(cur.fetchall())

web [('raw_stage/web_ee07ce1af1a3.json.gz', 'LOADED', 6, 6, 1, 0, None, None, None, None)]
mobile [('raw_stage/mobile_ee07ce1af1a3.json.gz', 'LOADED', 2, 2, 1, 0, None, None, None, None)]
[('web', 6), ('mobile', 2)]


In [22]:
INCLUDE_CURRENCY_CHECK = False

def build_sql_10_orders_clean(include_currency_check=False):

    currency_reason = (
        ", IFF(CURRENCY NOT IN ('INR','USD'), 'unsupported_currency', NULL)"
        if include_currency_check else ""
    )

    return f"""
    CREATE OR REPLACE TABLE FRESHCART_DB.SILVER.ORDERS_CHECKED AS

    WITH NORMALIZED AS (
        SELECT
            RAW_ID,
            SOURCE,
            BATCH_ID,
            INGESTED_AT,

            IFF(SOURCE='web',
                PAYLOAD:order_id::VARCHAR,
                PAYLOAD:orderId::VARCHAR) AS ORDER_ID,

            IFF(SOURCE='web',
                PAYLOAD:customer_id::VARCHAR,
                PAYLOAD:customerId::VARCHAR) AS CUSTOMER_ID,

            IFF(SOURCE='web',
                PAYLOAD:customer_email::VARCHAR,
                PAYLOAD:email::VARCHAR) AS CUSTOMER_EMAIL,

            IFF(SOURCE='web',
                PAYLOAD:order_date::DATE,
                PAYLOAD:orderDate::DATE) AS ORDER_DATE,

            IFF(SOURCE='web',
                PAYLOAD:amount::NUMBER(12,2),
                PAYLOAD:total::NUMBER(12,2)) AS AMOUNT,

            IFF(SOURCE='web',
                PAYLOAD:currency::VARCHAR,
                PAYLOAD:currencyCode::VARCHAR) AS CURRENCY,

            IFF(SOURCE='web',
                PAYLOAD:updated_at::TIMESTAMP_NTZ,
                PAYLOAD:updatedAt::TIMESTAMP_NTZ) AS UPDATED_AT

        FROM FRESHCART_DB.BRONZE.ORDERS_RAW
    ),

    REASONS AS (
        SELECT *,
            ARRAY_CONSTRUCT_COMPACT(
                IFF(ORDER_ID IS NULL, 'missing_order_id', NULL),
                IFF(CUSTOMER_ID IS NULL OR CUSTOMER_EMAIL IS NULL,
                    'missing_customer', NULL),
                IFF(AMOUNT < 0, 'negative_amount', NULL)
                {currency_reason}
            ) AS REASONS
        FROM NORMALIZED
    )

    SELECT * FROM REASONS
    """


SQL_10_ORDERS_CLEAN = build_sql_10_orders_clean(False)


def run_cleaning(include_currency_check=False):

    global SQL_10_ORDERS_CLEAN

    SQL_10_ORDERS_CLEAN = build_sql_10_orders_clean(
        include_currency_check
    )

    cur.execute(SQL_10_ORDERS_CLEAN)

    cur.execute("""
        CREATE OR REPLACE TABLE FRESHCART_DB.SILVER.ORDERS_CLEAN AS
        SELECT *
        FROM FRESHCART_DB.SILVER.ORDERS_CHECKED
        WHERE ARRAY_SIZE(REASONS)=0
    """)

    cur.execute("""
        CREATE OR REPLACE TABLE FRESHCART_DB.SILVER.ORDERS_QUARANTINE AS
        SELECT *
        FROM FRESHCART_DB.SILVER.ORDERS_CHECKED
        WHERE ARRAY_SIZE(REASONS)>0

        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY ORDER_ID
            ORDER BY INGESTED_AT DESC, RAW_ID DESC
        )=1
    """)

    print("Cleaning and quarantine completed")


run_cleaning(False)

Cleaning and quarantine completed


In [23]:
SQL_20_ORDERS_DEDUP = """
CREATE OR REPLACE TABLE FRESHCART_DB.SILVER.ORDERS_DEDUP AS

SELECT *
FROM FRESHCART_DB.SILVER.ORDERS_CLEAN

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY ORDER_ID
    ORDER BY UPDATED_AT DESC, RAW_ID DESC
)=1
"""

cur.execute(SQL_20_ORDERS_DEDUP)

print("Deduplication completed")

cur.execute("""
SELECT ORDER_ID, SOURCE, AMOUNT, UPDATED_AT
FROM SILVER.ORDERS_DEDUP
ORDER BY ORDER_ID
""")

display(pd.DataFrame(
    cur.fetchall(),
    columns=[c[0] for c in cur.description]
))

Deduplication completed


,ORDER_ID,SOURCE,AMOUNT,UPDATED_AT
0,ORD-1001,mobile,1100.00,2026-08-01 10:05:00
1,ORD-1002,web,785.00,2026-08-20 10:00:00
2,ORD-1003,mobile,2100.00,2026-08-10 10:30:00
3,ORD-1005,web,3800.00,2026-08-20 11:00:00


In [24]:
cur.execute("""
CREATE TABLE IF NOT EXISTS REF.MOCK_CRM_TIER_EVENTS (
    CUSTOMER_ID VARCHAR,
    CUSTOMER_EMAIL VARCHAR,
    TIER VARCHAR,
    EFFECTIVE_FROM DATE
)
""")

cur.execute("""
MERGE INTO REF.MOCK_CRM_TIER_EVENTS T

USING (
    SELECT * FROM VALUES
    ('CUST-01','priya.k@example.com','Silver','2026-01-01'),
    ('CUST-01','priya.k@example.com','Gold','2026-08-15'),
    ('CUST-02','arjun.k@yahoo.in','Silver','2026-01-01'),
    ('CUST-03','meera.s@example.com','Silver','2026-01-01')

    S(CUSTOMER_ID,CUSTOMER_EMAIL,TIER,EFFECTIVE_FROM)
) S

ON T.CUSTOMER_ID=S.CUSTOMER_ID
AND T.TIER=S.TIER
AND T.EFFECTIVE_FROM=TO_DATE(S.EFFECTIVE_FROM)

WHEN NOT MATCHED THEN
INSERT (
    CUSTOMER_ID,CUSTOMER_EMAIL,TIER,EFFECTIVE_FROM
)
VALUES (
    S.CUSTOMER_ID,S.CUSTOMER_EMAIL,S.TIER,TO_DATE(S.EFFECTIVE_FROM)
)
""")


def build_scd2():

    cur.execute("""
    CREATE OR REPLACE TABLE DIM.CUSTOMERS_SCD2 AS

    SELECT
        CUSTOMER_ID,
        CUSTOMER_EMAIL,
        TIER,
        EFFECTIVE_FROM,

        LEAD(EFFECTIVE_FROM) OVER (
            PARTITION BY CUSTOMER_ID
            ORDER BY EFFECTIVE_FROM
        ) AS EFFECTIVE_TO,

        IFF(
            LEAD(EFFECTIVE_FROM) OVER (
                PARTITION BY CUSTOMER_ID
                ORDER BY EFFECTIVE_FROM
            ) IS NULL,
            TRUE,
            FALSE
        ) AS IS_CURRENT

    FROM REF.MOCK_CRM_TIER_EVENTS
    """)


build_scd2()

cur.execute("""
SELECT *
FROM DIM.CUSTOMERS_SCD2
ORDER BY CUSTOMER_ID,EFFECTIVE_FROM
""")

display(pd.DataFrame(
    cur.fetchall(),
    columns=[c[0] for c in cur.description]
))

,CUSTOMER_ID,CUSTOMER_EMAIL,TIER,EFFECTIVE_FROM,EFFECTIVE_TO,IS_CURRENT
0,CUST-01,priya.k@example.com,Silver,2026-01-01,2026-08-15,False
1,CUST-01,priya.k@example.com,Gold,2026-08-15,None,True
2,CUST-02,arjun.k@yahoo.in,Silver,2026-01-01,None,True
3,CUST-03,meera.s@example.com,Silver,2026-01-01,None,True


In [25]:
cur.execute("""
CREATE TABLE IF NOT EXISTS GOLD.FCT_ORDERS (
    ORDER_ID VARCHAR,
    CUSTOMER_ID VARCHAR,
    ORDER_DATE DATE,
    AMOUNT_INR NUMBER(14,2),
    TIER VARCHAR,
    LOYALTY_POINTS NUMBER(14,0),
    ROW_HASH VARCHAR,
    PUBLISHED_AT TIMESTAMP_NTZ
)
""")


def build_gold():

    cur.execute("""
    CREATE OR REPLACE TEMP TABLE GOLD.STG_FCT_ORDERS AS

    WITH ENRICHED AS (

        SELECT
            O.ORDER_ID,
            O.CUSTOMER_ID,
            O.ORDER_DATE,

            ROUND(
                O.AMOUNT *
                CASE O.CURRENCY
                    WHEN 'INR' THEN 1.00
                    WHEN 'USD' THEN 83.50
                    ELSE NULL
                END, 2
            ) AS AMOUNT_INR,

            D.TIER,

            CASE D.TIER
                WHEN 'Silver' THEN 1.00
                WHEN 'Gold' THEN 2.25
                ELSE 0.50
            END AS POINT_MULTIPLIER

        FROM SILVER.ORDERS_DEDUP O

        JOIN DIM.CUSTOMERS_SCD2 D
          ON O.CUSTOMER_ID=D.CUSTOMER_ID
         AND O.ORDER_DATE >= D.EFFECTIVE_FROM
         AND (
             O.ORDER_DATE < D.EFFECTIVE_TO
             OR D.EFFECTIVE_TO IS NULL
         )
    ),

    POINTS AS (
        SELECT *,
            ROUND(
                AMOUNT_INR / 100 * POINT_MULTIPLIER,
                0
            ) AS LOYALTY_POINTS
        FROM ENRICHED
    )

    SELECT
        ORDER_ID,
        CUSTOMER_ID,
        ORDER_DATE,
        AMOUNT_INR,
        TIER,
        LOYALTY_POINTS,

        SHA2(
            CONCAT_WS(
                '|',
                ORDER_ID,
                CUSTOMER_ID,
                ORDER_DATE,
                AMOUNT_INR,
                TIER,
                LOYALTY_POINTS
            ),256
        ) AS ROW_HASH

    FROM POINTS
    """)

    cur.execute("""
    MERGE INTO GOLD.FCT_ORDERS T

    USING GOLD.STG_FCT_ORDERS S

    ON T.ORDER_ID=S.ORDER_ID

    WHEN MATCHED AND T.ROW_HASH <> S.ROW_HASH
    THEN UPDATE SET
        CUSTOMER_ID=S.CUSTOMER_ID,
        ORDER_DATE=S.ORDER_DATE,
        AMOUNT_INR=S.AMOUNT_INR,
        TIER=S.TIER,
        LOYALTY_POINTS=S.LOYALTY_POINTS,
        ROW_HASH=S.ROW_HASH,
        PUBLISHED_AT=CURRENT_TIMESTAMP()

    WHEN NOT MATCHED THEN
    INSERT (
        ORDER_ID,CUSTOMER_ID,ORDER_DATE,
        AMOUNT_INR,TIER,LOYALTY_POINTS,
        ROW_HASH,PUBLISHED_AT
    )
    VALUES (
        S.ORDER_ID,S.CUSTOMER_ID,S.ORDER_DATE,
        S.AMOUNT_INR,S.TIER,S.LOYALTY_POINTS,
        S.ROW_HASH,CURRENT_TIMESTAMP()
    )
    """)

    print("Gold table published")


build_gold()

Gold table published


In [26]:
@task(name="Clean orders", retries=1)
def clean_task():
    run_cleaning(INCLUDE_CURRENCY_CHECK)
    return "Cleaning completed"


@task(name="Deduplicate orders")
def dedup_task():
    cur.execute(SQL_20_ORDERS_DEDUP)
    return "Deduplication completed"


@task(name="Build customer SCD2")
def scd_task():
    build_scd2()
    return "SCD2 completed"


@task(name="Build Gold")
def gold_task():
    build_gold()
    return "Gold completed"


@flow(name="freshcart-elt-pipeline", log_prints=True)
def freshcart_pipeline():

    a = clean_task()
    b = dedup_task()
    c = scd_task()
    d = gold_task()

    print(a,b,c,d)

    return "FreshCart pipeline completed"


result = freshcart_pipeline()

print(result)

INFO:prefect:Starting temporary server on http://127.0.0.1:8982
See https://www.prefect.io/docs/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.
INFO:prefect.flow_runs:Beginning flow run 'helpful-wallaby' for flow 'freshcart-elt-pipeline'
INFO:prefect.task_runs:Cleaning and quarantine completed
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Gold table published
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.flow_runs:Cleaning completed Deduplication completed SCD2 completed Gold completed
INFO:prefect.flow_runs:Finished in state Completed()


FreshCart pipeline completed


In [27]:
def query_df(sql, params=None):

    cur.execute(sql, params or ())

    return pd.DataFrame(
        cur.fetchall(),
        columns=[c[0] for c in cur.description]
    )


def quality_checks():

    cur.execute("""
        SELECT COUNT(*)
        FROM BRONZE.ORDERS_RAW
        WHERE BATCH_ID=%s
    """, (BATCH_ID,))

    raw_count = cur.fetchone()[0]

    cur.execute("""
        SELECT COUNT(*)
        FROM SILVER.ORDERS_CLEAN
        WHERE BATCH_ID=%s
    """, (BATCH_ID,))

    clean_count = cur.fetchone()[0]

    cur.execute("""
        SELECT COUNT(*)
        FROM SILVER.ORDERS_QUARANTINE
    """)

    quarantine_count = cur.fetchone()[0]

    cur.execute("""
        SELECT COUNT(*)
        FROM SILVER.ORDERS_DEDUP
    """)

    dedup_count = cur.fetchone()[0]

    cur.execute("""
        SELECT COUNT(*), SUM(AMOUNT_INR),
               SUM(LOYALTY_POINTS)
        FROM GOLD.FCT_ORDERS
    """)

    gold_count, revenue, points = cur.fetchone()

    rejected = round(
        quarantine_count / raw_count * 100, 2
    )

    results = {
        "Raw records": raw_count,
        "Clean records": clean_count,
        "Quarantine": quarantine_count,
        "Deduplicated": dedup_count,
        "Gold orders": gold_count,
        "Revenue": float(revenue or 0),
        "Loyalty points": int(points or 0),
        "Rejected percentage": rejected
    }

    expected = {
        "Raw records": 8,
        "Clean records": 6,
        "Quarantine": 2,
        "Deduplicated": 4,
        "Gold orders": 4,
        "Revenue": 7785.0,
        "Loyalty points": 126,
        "Rejected percentage": 25.0
    }

    report = pd.DataFrame([
        {
            "Metric": key,
            "Actual": results[key],
            "Expected": value,
            "Status": "PASS" if results[key] == value else "CHECK"
        }
        for key, value in expected.items()
    ])

    display(report)

    return results


BASELINE_RESULTS = quality_checks()

,Metric,Actual,Expected,Status
0,Raw records,8.0,8.0,PASS
1,Clean records,6.0,6.0,PASS
2,Quarantine,2.0,2.0,PASS
3,Deduplicated,4.0,4.0,PASS
4,Gold orders,4.0,4.0,PASS
5,Revenue,7785.0,7785.0,PASS
6,Loyalty points,126.0,126.0,PASS
7,Rejected percentage,25.0,25.0,PASS


In [28]:
print("BRONZE RAW")
display(query_df("""
SELECT SOURCE, BATCH_ID, PAYLOAD
FROM BRONZE.ORDERS_RAW
WHERE BATCH_ID=%s
ORDER BY SOURCE
""", (BATCH_ID,)))

print("SILVER CLEAN")
display(query_df("""
SELECT ORDER_ID, CUSTOMER_ID, AMOUNT,
       CURRENCY, REASONS
FROM SILVER.ORDERS_CLEAN
WHERE BATCH_ID=%s
""", (BATCH_ID,)))

print("QUARANTINE")
display(query_df("""
SELECT ORDER_ID, REASONS
FROM SILVER.ORDERS_QUARANTINE
"""))

print("DEDUPLICATED")
display(query_df("""
SELECT ORDER_ID, AMOUNT, UPDATED_AT
FROM SILVER.ORDERS_DEDUP
ORDER BY ORDER_ID
"""))

print("GOLD")
display(query_df("""
SELECT *
FROM GOLD.FCT_ORDERS
ORDER BY ORDER_ID
"""))

BRONZE RAW


,SOURCE,BATCH_ID,PAYLOAD
0,mobile,ee07ce1af1a3,"{\n ""currencyCode"": ""INR"",\n ""customerId"": ""..."
1,mobile,ee07ce1af1a3,"{\n ""currencyCode"": ""INR"",\n ""customerId"": ""..."
2,web,ee07ce1af1a3,"{\n ""amount"": 1100,\n ""currency"": ""INR"",\n ..."
3,web,ee07ce1af1a3,"{\n ""amount"": 785,\n ""currency"": ""INR"",\n ""..."
4,web,ee07ce1af1a3,"{\n ""currency"": ""INR"",\n ""customer_email"": ""..."
5,web,ee07ce1af1a3,"{\n ""amount"": 300,\n ""currency"": ""INR"",\n ""..."
6,web,ee07ce1af1a3,"{\n ""amount"": 3800,\n ""currency"": ""INR"",\n ..."
7,web,ee07ce1af1a3,"{\n ""amount"": -50,\n ""currency"": ""EUR"",\n ""..."


SILVER CLEAN


,ORDER_ID,CUSTOMER_ID,AMOUNT,CURRENCY,REASONS
0,ORD-1001,CUST-01,1100.00,INR,[]
1,ORD-1002,CUST-02,785.00,INR,[]
2,ORD-1003,CUST-03,None,INR,[]
3,ORD-1005,CUST-01,3800.00,INR,[]
4,ORD-1001,CUST-01,1100.00,INR,[]
5,ORD-1003,CUST-03,2100.00,INR,[]


QUARANTINE


,ORDER_ID,REASONS
0,ORD-1004,"[\n ""missing_customer""\n]"
1,ORD-1006,"[\n ""negative_amount""\n]"


DEDUPLICATED


,ORDER_ID,AMOUNT,UPDATED_AT
0,ORD-1001,1100.00,2026-08-01 10:05:00
1,ORD-1002,785.00,2026-08-20 10:00:00
2,ORD-1003,2100.00,2026-08-10 10:30:00
3,ORD-1005,3800.00,2026-08-20 11:00:00


GOLD


,ORDER_ID,CUSTOMER_ID,ORDER_DATE,AMOUNT_INR,TIER,LOYALTY_POINTS,ROW_HASH,PUBLISHED_AT
0,ORD-1001,CUST-01,2026-08-01,1100.00,Silver,11,0999884c3b73d191ce31a2e668b6c49e9358c714b95743...,2026-11-16 08:07:02.050
1,ORD-1002,CUST-02,2026-08-20,785.00,Silver,8,e6f392a6f786f08a5d73a22b491f4958a7062a64e866fc...,2026-11-16 08:07:02.050
2,ORD-1003,CUST-03,2026-08-10,2100.00,Silver,21,02328267a8a2ed8f95fd861c9666babbefcb2f0f23b57a...,2026-11-16 08:07:02.050
3,ORD-1005,CUST-01,2026-08-20,3800.00,Gold,86,cfdbaf705ab53316373c83e3e248513aebce2d210be462...,2026-11-16 08:07:02.050


In [29]:
print("ORD-1003 raw records")

display(query_df("""
SELECT SOURCE, PAYLOAD
FROM BRONZE.ORDERS_RAW
WHERE BATCH_ID=%s
AND (
    PAYLOAD:order_id::VARCHAR='ORD-1003'
    OR PAYLOAD:orderId::VARCHAR='ORD-1003'
)
""", (BATCH_ID,)))


print("ORD-1003 normalized records")

display(query_df("""
SELECT ORDER_ID, SOURCE, AMOUNT, UPDATED_AT
FROM SILVER.ORDERS_CHECKED
WHERE ORDER_ID='ORD-1003'
ORDER BY UPDATED_AT
"""))


print("ORD-1003 surviving record")

display(query_df("""
SELECT ORDER_ID, AMOUNT, UPDATED_AT
FROM SILVER.ORDERS_DEDUP
WHERE ORDER_ID='ORD-1003'
"""))


print("Priya tier comparison")

display(query_df("""
SELECT ORDER_ID, ORDER_DATE,
       AMOUNT_INR, TIER, LOYALTY_POINTS
FROM GOLD.FCT_ORDERS
WHERE ORDER_ID IN ('ORD-1001','ORD-1005')
ORDER BY ORDER_ID
"""))

ORD-1003 raw records


,SOURCE,PAYLOAD
0,web,"{\n ""currency"": ""INR"",\n ""customer_email"": ""..."
1,mobile,"{\n ""currencyCode"": ""INR"",\n ""customerId"": ""..."


ORD-1003 normalized records


,ORDER_ID,SOURCE,AMOUNT,UPDATED_AT
0,ORD-1003,web,None,2026-08-10 09:00:00
1,ORD-1003,mobile,2100.00,2026-08-10 10:30:00


ORD-1003 surviving record


,ORDER_ID,AMOUNT,UPDATED_AT
0,ORD-1003,2100.00,2026-08-10 10:30:00


Priya tier comparison


,ORDER_ID,ORDER_DATE,AMOUNT_INR,TIER,LOYALTY_POINTS
0,ORD-1001,2026-08-01,1100.00,Silver,11
1,ORD-1005,2026-08-20,3800.00,Gold,86


In [30]:
print("ORD-1006 BEFORE currency validation")

display(query_df("""
SELECT ORDER_ID, CURRENCY, REASONS
FROM SILVER.ORDERS_CHECKED
WHERE ORDER_ID='ORD-1006'
ORDER BY INGESTED_AT DESC
LIMIT 1
"""))


ORD-1006 BEFORE currency validation


,ORDER_ID,CURRENCY,REASONS
0,ORD-1006,EUR,"[\n ""negative_amount""\n]"


In [33]:
display(query_df("""
SELECT ORDER_ID, COUNT(*) AS ROW_COUNT
FROM SILVER.ORDERS_QUARANTINE
WHERE ORDER_ID='ORD-1006'
GROUP BY ORDER_ID
"""))

,ORDER_ID,ROW_COUNT
0,ORD-1006,1


In [36]:
import time
time.sleep(1)

after_scd = query_df("""
SELECT * FROM DIM.CUSTOMERS_SCD2
WHERE CUSTOMER_ID='CUST-02'
""")

after_order = query_df("""
SELECT ORDER_ID,TIER,LOYALTY_POINTS,
       ROW_HASH,PUBLISHED_AT
FROM GOLD.FCT_ORDERS
WHERE ORDER_ID='ORD-1002'
""")

after_total = query_df("""
SELECT SUM(LOYALTY_POINTS)
FROM GOLD.FCT_ORDERS
""").iloc[0,0]

after_dates = query_df("""
SELECT ORDER_ID,PUBLISHED_AT
FROM GOLD.FCT_ORDERS
""")

comparison = before_dates.merge(
    after_dates,
    on="ORDER_ID",
    suffixes=("_BEFORE","_AFTER")
)

changed = (
    comparison["PUBLISHED_AT_BEFORE"]
    != comparison["PUBLISHED_AT_AFTER"]
).sum()

print("After SCD2 rows:", len(after_scd))
display(after_order)

print("After total points:", after_total)
print("Published timestamps changed:", changed)

After SCD2 rows: 2


,ORDER_ID,TIER,LOYALTY_POINTS,ROW_HASH,PUBLISHED_AT
0,ORD-1002,Gold,18,8b27a62ae34735190f9a2abe1bc52524b001aec783b2e5...,2026-11-16 08:14:00.284


After total points: 136
Published timestamps changed: 1


In [37]:
before_repeat = query_df("""
SELECT ORDER_ID,ROW_HASH,PUBLISHED_AT
FROM GOLD.FCT_ORDERS
ORDER BY ORDER_ID
""")

build_scd2()
cur.execute(SQL_20_ORDERS_DEDUP)
build_gold()

after_repeat = query_df("""
SELECT ORDER_ID,ROW_HASH,PUBLISHED_AT
FROM GOLD.FCT_ORDERS
ORDER BY ORDER_ID
""")

print("Gold rows before:", len(before_repeat))
print("Gold rows after:", len(after_repeat))

print(
    "Hashes unchanged:",
    before_repeat["ROW_HASH"].equals(
        after_repeat["ROW_HASH"]
    )
)

print(
    "Published timestamps unchanged:",
    before_repeat["PUBLISHED_AT"].equals(
        after_repeat["PUBLISHED_AT"]
    )
)

Gold table published
Gold rows before: 4
Gold rows after: 4
Hashes unchanged: True
Published timestamps unchanged: True


In [38]:
def sixth_quality_check():

    cur.execute("""
        SELECT COUNT(*)
        FROM GOLD.FCT_ORDERS
        WHERE LOYALTY_POINTS < 0
    """)

    count = cur.fetchone()[0]

    status = "PASS" if count == 0 else "PROBLEM"

    print(
        "Negative loyalty points:",
        count,
        status
    )

    return count


print("Normal data:")
assert sixth_quality_check() == 0

Normal data:
Negative loyalty points: 0 PASS


In [40]:
cur.execute("""
DELETE FROM GOLD.FCT_ORDERS
WHERE ORDER_ID='MOCK-NEGATIVE'
""")

print("After removing invalid record:")

assert sixth_quality_check() == 0

After removing invalid record:
Negative loyalty points: 0 PASS


In [41]:
print("FINAL BRONZE")
display(query_df("""
SELECT COUNT(*) AS RAW_ROWS
FROM BRONZE.ORDERS_RAW
"""))

print("FINAL SILVER")
display(query_df("""
SELECT COUNT(*) AS CLEAN_ROWS
FROM SILVER.ORDERS_CLEAN
"""))

print("QUARANTINE")
display(query_df("""
SELECT ORDER_ID,REASONS
FROM SILVER.ORDERS_QUARANTINE
"""))

print("DEDUPLICATION")
display(query_df("""
SELECT COUNT(*) AS UNIQUE_ORDERS
FROM SILVER.ORDERS_DEDUP
"""))

print("FINAL GOLD")
display(query_df("""
SELECT
    COUNT(*) AS TOTAL_ORDERS,
    SUM(AMOUNT_INR) AS REVENUE,
    SUM(LOYALTY_POINTS) AS LOYALTY_POINTS
FROM GOLD.FCT_ORDERS
"""))

FINAL BRONZE


,RAW_ROWS
0,16


FINAL SILVER


,CLEAN_ROWS
0,12


QUARANTINE


,ORDER_ID,REASONS
0,ORD-1004,"[\n ""missing_customer""\n]"
1,ORD-1006,"[\n ""negative_amount"",\n ""unsupported_curren..."


DEDUPLICATION


,UNIQUE_ORDERS
0,4


FINAL GOLD


,TOTAL_ORDERS,REVENUE,LOYALTY_POINTS
0,4,7785.00,136
